In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="multimodal",
    notebook_name="01_vision_language_experiments.ipynb"
)

# Experiments: Vision-Language Models

This notebook produces **runnable evidence** for claims you will make in interviews about CLIP and contrastive learning. Each experiment tests a specific claim and gives you real numbers to cite.

Before running this notebook, make sure you have read [vision-language README](./README.md) and worked through [01_vision_language.ipynb](./01_vision_language.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time

matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")


# ---- Shared utilities from the concept notebook ----

def cosine_similarity_matrix(A, B):
    """Compute NxM cosine similarity matrix between rows of A and B."""
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def infonce_loss(image_embeds, text_embeds, temperature=0.07):
    N = len(image_embeds)
    sim = cosine_similarity_matrix(image_embeds, text_embeds)
    logits = sim / temperature
    labels = np.arange(N)
    probs_i2t = softmax(logits, axis=1)
    loss_i2t = -np.mean(np.log(probs_i2t[np.arange(N), labels] + 1e-8))
    probs_t2i = softmax(logits, axis=0)
    loss_t2i = -np.mean(np.log(probs_t2i[labels, np.arange(N)] + 1e-8))
    return (loss_i2t + loss_t2i) / 2

# Create synthetic concept data
n_concepts = 16
embed_dim = 32
concept_directions = np.random.randn(n_concepts, embed_dim)
concept_directions = concept_directions / np.linalg.norm(concept_directions, axis=1, keepdims=True)

def make_batch(batch_size, noise_level=0.3):
    indices = np.random.choice(n_concepts, size=batch_size, replace=False)
    images = concept_directions[indices] + np.random.randn(batch_size, embed_dim) * noise_level
    texts = concept_directions[indices] + np.random.randn(batch_size, embed_dim) * noise_level
    return images, texts, indices

def train_model(W_img, W_txt, n_steps=100, batch_size=8, temperature=0.1, lr=0.3, noise=0.3):
    """Train with numerical gradients. Returns loss history."""
    losses = []
    eps = 1e-4
    for step in range(n_steps):
        raw_img, raw_txt, _ = make_batch(batch_size, noise_level=noise)
        img_enc = raw_img @ W_img
        img_enc = img_enc / np.linalg.norm(img_enc, axis=1, keepdims=True)
        txt_enc = raw_txt @ W_txt
        txt_enc = txt_enc / np.linalg.norm(txt_enc, axis=1, keepdims=True)
        loss = infonce_loss(img_enc, txt_enc, temperature=temperature)
        losses.append(loss)
        
        # Gradient for W_img
        grad_img = np.zeros_like(W_img)
        for i in range(W_img.shape[0]):
            for j in range(W_img.shape[1]):
                W_img[i,j] += eps
                e = raw_img @ W_img; e = e / np.linalg.norm(e, axis=1, keepdims=True)
                lp = infonce_loss(e, txt_enc, temperature=temperature)
                W_img[i,j] -= 2*eps
                e = raw_img @ W_img; e = e / np.linalg.norm(e, axis=1, keepdims=True)
                lm = infonce_loss(e, txt_enc, temperature=temperature)
                W_img[i,j] += eps
                grad_img[i,j] = (lp - lm) / (2*eps)
        
        grad_txt = np.zeros_like(W_txt)
        for i in range(W_txt.shape[0]):
            for j in range(W_txt.shape[1]):
                W_txt[i,j] += eps
                e = raw_txt @ W_txt; e = e / np.linalg.norm(e, axis=1, keepdims=True)
                lp = infonce_loss(img_enc, e, temperature=temperature)
                W_txt[i,j] -= 2*eps
                e = raw_txt @ W_txt; e = e / np.linalg.norm(e, axis=1, keepdims=True)
                lm = infonce_loss(img_enc, e, temperature=temperature)
                W_txt[i,j] += eps
                grad_txt[i,j] = (lp - lm) / (2*eps)
        
        W_img -= lr * grad_img
        W_txt -= lr * grad_txt
    
    return losses, W_img, W_txt

print(f"Concepts: {n_concepts}, Embedding dim: {embed_dim}")

## Experiment 1: Temperature Ablation

**Claim:** Temperature τ controls the sharpness of the contrastive loss. Too small → training collapses (gradients only flow through the single hardest negative). Too large → no discrimination (all pairs look equally similar).

**Why it matters in an interview:** Interviewers ask "why does CLIP learn τ?" The answer is that the optimal τ depends on the data distribution and training stage. This experiment shows what happens at extreme values.

In [ ]:
# Train models at different temperatures
temperatures = [0.01, 0.05, 0.1, 0.5, 1.0]
temp_results = {}
output_dim = 8

print("Training models at different temperatures...")
for temp in temperatures:
    np.random.seed(42)
    W_img = np.random.randn(embed_dim, output_dim) * 0.1
    W_txt = np.random.randn(embed_dim, output_dim) * 0.1
    losses, W_img_final, W_txt_final = train_model(
        W_img.copy(), W_txt.copy(), n_steps=80, batch_size=8,
        temperature=temp, lr=0.3
    )
    temp_results[temp] = {
        'losses': losses,
        'final_loss': losses[-1],
        'W_img': W_img_final,
        'W_txt': W_txt_final
    }
    print(f"  τ = {temp:.2f}: final loss = {losses[-1]:.4f}")

print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Training curves
colors = plt.cm.viridis(np.linspace(0, 0.9, len(temperatures)))
for temp, color in zip(temperatures, colors):
    losses = temp_results[temp]['losses']
    axes[0].plot(losses, color=color, linewidth=2, label=f'τ={temp}', alpha=0.8)

axes[0].set_xlabel('Training Step', fontsize=13)
axes[0].set_ylabel('InfoNCE Loss', fontsize=13)
axes[0].set_title('Training Curves at Different Temperatures', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right: Final loss vs temperature
final_losses = [temp_results[t]['final_loss'] for t in temperatures]
axes[1].plot(temperatures, final_losses, 'o-', color='#2196F3', linewidth=2, markersize=10)
best_idx = np.argmin(final_losses)
axes[1].scatter([temperatures[best_idx]], [final_losses[best_idx]],
                color='red', s=200, zorder=5, label=f'Best: τ={temperatures[best_idx]}')
axes[1].set_xlabel('Temperature (τ)', fontsize=13)
axes[1].set_ylabel('Final Loss', fontsize=13)
axes[1].set_title('Final Loss vs Temperature', fontsize=14)
axes[1].set_xscale('log')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Also show softmax sharpness at each temperature
print("\nSoftmax sharpness demo (same similarities, different τ):")
sims = np.array([0.8, 0.5, 0.3, 0.1, -0.1, -0.3])
print(f"  Raw similarities: {sims}")
for temp in temperatures:
    probs = softmax(sims / temp)
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    print(f"  τ={temp:>4}: max_prob={probs[0]:.4f}, entropy={entropy:.4f} "
          f"({'too sharp' if entropy < 0.5 else 'too flat' if entropy > 1.5 else 'good range'})")

print("\n▶ Interview sentence: 'Temperature controls the effective number of negatives.")
print("  At τ=0.01, only ~1 negative contributes gradient. At τ=0.1, ~3-5 contribute.")
print("  CLIP learns τ to adapt to the training data distribution.'")

## Experiment 2: Embedding Collapse

**Claim:** Without contrastive loss, embeddings collapse to a constant vector. The model finds a shortcut: if all image embeddings are the same point, and all text embeddings are the same point, the loss can appear low, but the model is useless.

**Why it matters in an interview:** Shows you understand why contrastive training is designed the way it is. The loss function is specifically crafted to prevent this collapse.

In [ ]:
# Demonstrate embedding collapse: train with a "broken" loss
# that only tries to make all embeddings similar (no negatives)

def broken_loss_no_negatives(image_embeds, text_embeds):
    """Only maximize similarity of matching pairs. No negative pairs.
    This causes collapse because the easiest way to maximize all
    cosine similarities is to make everything the same vector."""
    img_norm = image_embeds / np.linalg.norm(image_embeds, axis=1, keepdims=True)
    txt_norm = text_embeds / np.linalg.norm(text_embeds, axis=1, keepdims=True)
    # Only diagonal (matching pairs)
    diag_sims = np.sum(img_norm * txt_norm, axis=1)
    return -np.mean(diag_sims)  # Minimize negative similarity = maximize similarity


# Train with broken loss (no negatives)
np.random.seed(42)
W_img_broken = np.random.randn(embed_dim, output_dim) * 0.1
W_txt_broken = np.random.randn(embed_dim, output_dim) * 0.1

collapse_stds_img = []
collapse_stds_txt = []
collapse_losses = []
n_collapse_steps = 100
eps = 1e-4

print("Training with broken loss (no negatives — should collapse)...")
for step in range(n_collapse_steps):
    raw_img, raw_txt, _ = make_batch(8, noise_level=0.3)
    img_enc = raw_img @ W_img_broken
    img_enc = img_enc / np.linalg.norm(img_enc, axis=1, keepdims=True)
    txt_enc = raw_txt @ W_txt_broken
    txt_enc = txt_enc / np.linalg.norm(txt_enc, axis=1, keepdims=True)
    
    loss = broken_loss_no_negatives(img_enc, txt_enc)
    collapse_losses.append(loss)
    
    # Track embedding diversity (std across batch)
    collapse_stds_img.append(np.std(img_enc))
    collapse_stds_txt.append(np.std(txt_enc))
    
    # Numerical gradient for W_img_broken
    grad = np.zeros_like(W_img_broken)
    for i in range(W_img_broken.shape[0]):
        for j in range(W_img_broken.shape[1]):
            W_img_broken[i,j] += eps
            e = raw_img @ W_img_broken
            e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lp = broken_loss_no_negatives(e, txt_enc)
            W_img_broken[i,j] -= 2*eps
            e = raw_img @ W_img_broken
            e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lm = broken_loss_no_negatives(e, txt_enc)
            W_img_broken[i,j] += eps
            grad[i,j] = (lp - lm) / (2*eps)
    W_img_broken -= 0.5 * grad
    
    # Same for W_txt_broken
    grad = np.zeros_like(W_txt_broken)
    img_enc_fixed = raw_img @ W_img_broken
    img_enc_fixed = img_enc_fixed / np.linalg.norm(img_enc_fixed, axis=1, keepdims=True)
    for i in range(W_txt_broken.shape[0]):
        for j in range(W_txt_broken.shape[1]):
            W_txt_broken[i,j] += eps
            e = raw_txt @ W_txt_broken
            e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lp = broken_loss_no_negatives(img_enc_fixed, e)
            W_txt_broken[i,j] -= 2*eps
            e = raw_txt @ W_txt_broken
            e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lm = broken_loss_no_negatives(img_enc_fixed, e)
            W_txt_broken[i,j] += eps
            grad[i,j] = (lp - lm) / (2*eps)
    W_txt_broken -= 0.5 * grad
    
    if step % 25 == 0:
        print(f"  Step {step:>3d}: loss={loss:.4f}, img_std={collapse_stds_img[-1]:.4f}, txt_std={collapse_stds_txt[-1]:.4f}")

print(f"\nFinal: img_std={collapse_stds_img[-1]:.4f}, txt_std={collapse_stds_txt[-1]:.4f}")
print("If std is very low, all embeddings have collapsed to the same point.")

In [ ]:
# Now train with proper contrastive loss for comparison
np.random.seed(42)
W_img_good = np.random.randn(embed_dim, output_dim) * 0.1
W_txt_good = np.random.randn(embed_dim, output_dim) * 0.1

good_stds_img = []
good_stds_txt = []

print("Training with proper contrastive loss (should NOT collapse)...")
for step in range(n_collapse_steps):
    raw_img, raw_txt, _ = make_batch(8, noise_level=0.3)
    img_enc = raw_img @ W_img_good
    img_enc = img_enc / np.linalg.norm(img_enc, axis=1, keepdims=True)
    txt_enc = raw_txt @ W_txt_good
    txt_enc = txt_enc / np.linalg.norm(txt_enc, axis=1, keepdims=True)
    
    good_stds_img.append(np.std(img_enc))
    good_stds_txt.append(np.std(txt_enc))
    
    loss = infonce_loss(img_enc, txt_enc, temperature=0.1)
    
    grad = np.zeros_like(W_img_good)
    for i in range(W_img_good.shape[0]):
        for j in range(W_img_good.shape[1]):
            W_img_good[i,j] += eps
            e = raw_img @ W_img_good; e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lp = infonce_loss(e, txt_enc, temperature=0.1)
            W_img_good[i,j] -= 2*eps
            e = raw_img @ W_img_good; e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lm = infonce_loss(e, txt_enc, temperature=0.1)
            W_img_good[i,j] += eps
            grad[i,j] = (lp - lm) / (2*eps)
    W_img_good -= 0.3 * grad
    
    grad = np.zeros_like(W_txt_good)
    for i in range(W_txt_good.shape[0]):
        for j in range(W_txt_good.shape[1]):
            W_txt_good[i,j] += eps
            e = raw_txt @ W_txt_good; e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lp = infonce_loss(img_enc, e, temperature=0.1)
            W_txt_good[i,j] -= 2*eps
            e = raw_txt @ W_txt_good; e = e / np.linalg.norm(e, axis=1, keepdims=True)
            lm = infonce_loss(img_enc, e, temperature=0.1)
            W_txt_good[i,j] += eps
            grad[i,j] = (lp - lm) / (2*eps)
    W_txt_good -= 0.3 * grad

print(f"Final: img_std={good_stds_img[-1]:.4f}, txt_std={good_stds_txt[-1]:.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(collapse_stds_img, color='#F44336', linewidth=2, label='Image (broken loss)')
axes[0].plot(collapse_stds_txt, color='#FF9800', linewidth=2, label='Text (broken loss)', linestyle='--')
axes[0].plot(good_stds_img, color='#4CAF50', linewidth=2, label='Image (contrastive)')
axes[0].plot(good_stds_txt, color='#2196F3', linewidth=2, label='Text (contrastive)', linestyle='--')
axes[0].set_xlabel('Training Step', fontsize=13)
axes[0].set_ylabel('Embedding Std (diversity)', fontsize=13)
axes[0].set_title('Embedding Diversity Over Training\n(lower = more collapsed)', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Show final embeddings
raw_test, _, _ = make_batch(8, noise_level=0.2)
collapsed_enc = raw_test @ W_img_broken
collapsed_enc = collapsed_enc / np.linalg.norm(collapsed_enc, axis=1, keepdims=True)
good_enc = raw_test @ W_img_good
good_enc = good_enc / np.linalg.norm(good_enc, axis=1, keepdims=True)

# PCA to 2D for both
def pca_2d(X):
    centered = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ Vt[:2].T

col_2d = pca_2d(collapsed_enc)
good_2d = pca_2d(good_enc)

axes[1].scatter(col_2d[:, 0], col_2d[:, 1], c='#F44336', s=100, marker='x',
                linewidth=2, label='Broken loss (collapsed)', zorder=5)
axes[1].scatter(good_2d[:, 0], good_2d[:, 1], c='#4CAF50', s=100, marker='o',
                edgecolors='black', label='Contrastive (diverse)', zorder=5)
axes[1].set_xlabel('PCA 1', fontsize=13)
axes[1].set_ylabel('PCA 2', fontsize=13)
axes[1].set_title('Final Embeddings (PCA)\nCollapsed vs Diverse', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n▶ Interview sentence: 'Without negative pairs in the loss, embeddings")
print("  collapse to a constant vector. Contrastive loss prevents this by requiring")
print("  the model to distinguish matching from non-matching pairs.'")

## Experiment 3: Batch Size Effect

**Claim:** Larger batch sizes provide more negative pairs per sample, leading to better contrastive learning. CLIP uses batch size 32,768 for this reason.

**Why it matters in an interview:** This is the connection between InfoNCE and mutual information. The InfoNCE loss is a lower bound on MI: I(image; text) >= log(N) - L. Larger N = tighter bound = more learnable information.

In [ ]:
# Measure contrastive loss quality at different batch sizes
batch_sizes = [2, 4, 6, 8, 10, 12, 14]
batch_size_results = {}

print("Training at different batch sizes...")
for bs in batch_sizes:
    np.random.seed(42)
    W_img = np.random.randn(embed_dim, output_dim) * 0.1
    W_txt = np.random.randn(embed_dim, output_dim) * 0.1
    losses, W_img_f, W_txt_f = train_model(
        W_img.copy(), W_txt.copy(), n_steps=80, batch_size=bs,
        temperature=0.1, lr=0.3
    )
    
    # Measure zero-shot accuracy on all concepts
    cat_txt = concept_directions @ W_txt_f
    cat_txt = cat_txt / np.linalg.norm(cat_txt, axis=1, keepdims=True)
    correct = 0
    for i in range(n_concepts):
        test_img = concept_directions[i] + np.random.randn(embed_dim) * 0.15
        test_enc = (test_img @ W_img_f).reshape(1, -1)
        test_enc = test_enc / np.linalg.norm(test_enc)
        sims = test_enc @ cat_txt.T
        if np.argmax(sims) == i:
            correct += 1
    acc = correct / n_concepts
    
    batch_size_results[bs] = {
        'losses': losses,
        'final_loss': losses[-1],
        'accuracy': acc
    }
    print(f"  Batch size {bs:>2d}: final_loss={losses[-1]:.4f}, accuracy={acc:.0%} ({correct}/{n_concepts})")

print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Training curves
colors = plt.cm.plasma(np.linspace(0, 0.9, len(batch_sizes)))
for bs, color in zip(batch_sizes, colors):
    axes[0].plot(batch_size_results[bs]['losses'], color=color, linewidth=1.5,
                label=f'N={bs}', alpha=0.8)
axes[0].set_xlabel('Training Step', fontsize=13)
axes[0].set_ylabel('InfoNCE Loss', fontsize=13)
axes[0].set_title('Training Curves at Different Batch Sizes', fontsize=14)
axes[0].legend(fontsize=10, ncol=2)
axes[0].grid(True, alpha=0.3)

# Right: Accuracy vs batch size
accs = [batch_size_results[bs]['accuracy'] for bs in batch_sizes]
axes[1].plot(batch_sizes, accs, 'o-', color='#4CAF50', linewidth=2, markersize=10)
axes[1].set_xlabel('Batch Size (N)', fontsize=13)
axes[1].set_ylabel('Zero-Shot Accuracy', fontsize=13)
axes[1].set_title('Zero-Shot Accuracy vs Batch Size\n(more negatives = better learning)', fontsize=14)
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3)

# Add MI bound annotation
for bs in batch_sizes:
    axes[1].annotate(f'{batch_size_results[bs]["accuracy"]:.0%}',
                     (bs, batch_size_results[bs]['accuracy']),
                     textcoords='offset points', xytext=(0, 12),
                     fontsize=10, ha='center')

plt.tight_layout()
plt.show()

print("\n▶ Interview sentence: 'InfoNCE loss is a lower bound on mutual information:")
print("  I(image;text) >= log(N) - L. Larger batch size N raises the ceiling on")
print("  learnable information. CLIP uses N=32,768 to maximize this bound.'")

## Experiment 4: Modality Gap

**Claim:** Even after training, CLIP image and text embeddings form two separate clusters in embedding space. The gap does not hurt zero-shot classification (relative rankings are preserved) but hurts cross-modal retrieval.

**Why it matters in an interview:** Shows understanding of a real limitation in production CLIP systems. The fix (post-hoc centering) is simple but non-obvious.

In [ ]:
# Train a model, then measure the modality gap
np.random.seed(42)
W_img_gap = np.random.randn(embed_dim, output_dim) * 0.1
W_txt_gap = np.random.randn(embed_dim, output_dim) * 0.1

print("Training model to measure modality gap...")
losses_gap, W_img_gap, W_txt_gap = train_model(
    W_img_gap.copy(), W_txt_gap.copy(), n_steps=150, batch_size=10,
    temperature=0.1, lr=0.3
)
print(f"Final loss: {losses_gap[-1]:.4f}")

# Encode all concepts
all_img_raw = concept_directions + np.random.randn(n_concepts, embed_dim) * 0.1
all_txt_raw = concept_directions + np.random.randn(n_concepts, embed_dim) * 0.1

all_img_enc = all_img_raw @ W_img_gap
all_img_enc = all_img_enc / np.linalg.norm(all_img_enc, axis=1, keepdims=True)
all_txt_enc = all_txt_raw @ W_txt_gap
all_txt_enc = all_txt_enc / np.linalg.norm(all_txt_enc, axis=1, keepdims=True)

# Measure the gap
mean_img = all_img_enc.mean(axis=0)
mean_txt = all_txt_enc.mean(axis=0)
mean_img_norm = mean_img / np.linalg.norm(mean_img)
mean_txt_norm = mean_txt / np.linalg.norm(mean_txt)
gap_cos = np.dot(mean_img_norm, mean_txt_norm)

print(f"\nModality Gap Measurement:")
print(f"  Cosine similarity between mean image and mean text embedding: {gap_cos:.4f}")
print(f"  (1.0 = perfectly overlapping, 0.0 = orthogonal, -1.0 = opposite)")
if gap_cos < 0.7:
    print(f"  GAP DETECTED: image and text clusters are separated")
else:
    print(f"  Clusters are reasonably close")

In [ ]:
# Visualize the modality gap
all_embeds = np.vstack([all_img_enc, all_txt_enc])
proj_2d = pca_2d(all_embeds)
img_2d = proj_2d[:n_concepts]
txt_2d = proj_2d[n_concepts:]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Show modality gap
axes[0].scatter(img_2d[:, 0], img_2d[:, 1], c='#2196F3', s=120, marker='o',
                edgecolors='black', label='Image embeddings', zorder=5)
axes[0].scatter(txt_2d[:, 0], txt_2d[:, 1], c='#F44336', s=120, marker='^',
                edgecolors='black', label='Text embeddings', zorder=5)

# Draw connections for matching pairs
for i in range(n_concepts):
    axes[0].plot([img_2d[i, 0], txt_2d[i, 0]], [img_2d[i, 1], txt_2d[i, 1]],
                 'k--', alpha=0.2, linewidth=1)

# Mark cluster centers
img_center = img_2d.mean(axis=0)
txt_center = txt_2d.mean(axis=0)
axes[0].scatter(*img_center, c='#2196F3', s=300, marker='*', edgecolors='black',
                linewidth=2, zorder=10, label='Image center')
axes[0].scatter(*txt_center, c='#F44336', s=300, marker='*', edgecolors='black',
                linewidth=2, zorder=10, label='Text center')
axes[0].annotate('', xy=txt_center, xytext=img_center,
                 arrowprops=dict(arrowstyle='->', color='black', lw=2))
axes[0].set_title(f'Modality Gap (cos={gap_cos:.3f})\nImage and text clusters are separated', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Show effect of centering fix
centered_img = all_img_enc - mean_img
centered_txt = all_txt_enc - mean_txt
centered_img = centered_img / np.linalg.norm(centered_img, axis=1, keepdims=True)
centered_txt = centered_txt / np.linalg.norm(centered_txt, axis=1, keepdims=True)

all_centered = np.vstack([centered_img, centered_txt])
proj_centered = pca_2d(all_centered)
cimg_2d = proj_centered[:n_concepts]
ctxt_2d = proj_centered[n_concepts:]

axes[1].scatter(cimg_2d[:, 0], cimg_2d[:, 1], c='#2196F3', s=120, marker='o',
                edgecolors='black', label='Image (centered)', zorder=5)
axes[1].scatter(ctxt_2d[:, 0], ctxt_2d[:, 1], c='#F44336', s=120, marker='^',
                edgecolors='black', label='Text (centered)', zorder=5)
for i in range(n_concepts):
    axes[1].plot([cimg_2d[i, 0], ctxt_2d[i, 0]], [cimg_2d[i, 1], ctxt_2d[i, 1]],
                 'k--', alpha=0.2, linewidth=1)

new_gap = np.dot(
    centered_img.mean(axis=0) / np.linalg.norm(centered_img.mean(axis=0)),
    centered_txt.mean(axis=0) / np.linalg.norm(centered_txt.mean(axis=0))
)
axes[1].set_title(f'After Centering Fix\nGap reduced', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Measure retrieval accuracy before and after centering
def retrieval_accuracy(img_enc, txt_enc):
    sim = cosine_similarity_matrix(img_enc, txt_enc)
    correct = sum(np.argmax(sim[i]) == i for i in range(len(sim)))
    return correct / len(sim)

acc_before = retrieval_accuracy(all_img_enc, all_txt_enc)
acc_after = retrieval_accuracy(centered_img, centered_txt)
print(f"\nRetrieval accuracy (image → text):")
print(f"  Before centering: {acc_before:.0%}")
print(f"  After centering:  {acc_after:.0%}")

print("\n▶ Interview sentence: 'CLIP embeddings have a modality gap — image and text")
print("  clusters are systematically offset. Zero-shot classification is unaffected")
print("  because it compares within the text modality. Cross-modal retrieval is hurt.")
print("  Post-hoc centering (subtract modality means) improves retrieval by 2-5%.'")

## Summary: Claims Now Backed by Evidence

| Experiment | Claim | Evidence |
|-----------|-------|----------|
| Temperature ablation | τ controls effective number of negatives | Loss curves diverge at extreme τ; optimal τ is in a narrow range |
| Embedding collapse | Without negatives, all embeddings converge | Std drops to near-zero without contrastive loss |
| Batch size effect | Larger N = better learning | Zero-shot accuracy increases with batch size |
| Modality gap | Image/text clusters are offset | Cluster centers have low cosine similarity; centering fixes retrieval |

For the full mathematical treatment and interview Q&A, see [vision-language-interview.md](./vision-language-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)